# Lab 1 — 첫 LLM 호출 & 에이전트의 뼈대

> **이론 복습 — Session 1 슬라이드**
> - LLM 은 "다음 토큰 예측기" 다. **텍스트만** 생성한다.
> - LLM 의 3가지 한계: ① 환각  ② 멈춰 있는 지식  ③ 행동 불가
> - 에이전트 = **LLM + Tools + Loop + Instructions(System Prompt)**

## lab0 와의 연결 — 양파 까기

lab0 에서 만든 챗봇은 위 4개를 모두 갖춘 **진짜 에이전트** 다.
`example_solution/src/lib/` 안을 한 파일씩 보면 한 개념씩 들어 있다:

| lab0 파일 | 무엇 | 어느 lab 에서 뜯어보나 |
|---|---|---|
| `lib/chat.js` ▶ `SYSTEM_PROMPT` + `callGemini()` | **LLM 호출 + Instructions** | **이번 lab 1** |
| `lib/tools.js` ▶ 도구 5개 | **Tools** | lab 2 |
| `lib/agent.js` ▶ `runAgent()` | **Loop** | lab 3 |

lab1 은 가장 안쪽 — **LLM 자체** 만 다룬다. 도구와 루프는 다음 lab.

## 학습 목표
1. `common/llm.py` 로 LLM 을 호출한다 — lab0 의 `callGemini()` 와 같은 역할
2. LLM 의 한계 ①②③ 을 **lab0 통근 데이터로** 직접 재현한다
3. lab0 의 `SYSTEM_PROMPT` 가 무엇을 하고, 무엇을 *못* 하는지 확인한다
4. 도구 없이, 프롬프트만으로 "추론 → 행동 제안" 형식을 끌어낸다 (ReAct 맛보기 → lab2 의 씨앗)


## 0. 준비

- `pip install -r requirements.txt` 가 끝나 있어야 합니다.
- `.env` 에 `GEMINI_API_KEY` 가 있으면 **실제 모델**, 없으면 `MockLLM` 으로 동작합니다.
- 이 노트북은 `labs/` 폴더에서 실행하세요.


In [ ]:
import json
from pathlib import Path
from common.llm import LLMClient

llm = LLMClient()
print("mock 모드인가요?", llm.is_mock)

# lab0 에서 만든 챗봇이 쓰던 그 데이터를 그대로 읽는다.
DATA = Path("lab0_vibe_coding/data")
cities = json.loads((DATA / "cities.json").read_text(encoding="utf-8"))
flows  = json.loads((DATA / "flows.json").read_text(encoding="utf-8"))

print(f"도시 {len(cities)}개, 통근 흐름 {len(flows)}개")
print("샘플 도시:", cities[0])
print("샘플 흐름:", flows[0])

## 1. 첫 LLM 호출

가장 단순한 사용법입니다. 문자열을 넣으면 `LLMResponse` 가 돌아옵니다.
`.text` 에 모델의 답변이 들어 있습니다.

이게 lab0 의 `callGemini()` 가 내부에서 하는 일의 본질입니다 — *문자열 보내고, 문자열 받기*.


In [ ]:
reply = llm.generate("Agentic AI 를 학부생에게 한 문장으로 설명해줘.")
print(reply.text)

## 2. LLM 의 한계를 lab0 데이터로 재현하기

lab0 의 챗봇이 *왜 도구를 써야 했는지* 를 알려면, 먼저 **도구 없이** 같은 질문을 던져 봐야 합니다.
슬라이드의 3가지 한계가 어떻게 나타나는지 관찰하세요.


In [ ]:
# 한계 ① — 환각: 데이터를 안 줬는데 그럴듯한 숫자를 만들어 낸다.
print(llm.generate(
    "우리 수도권 통근 데이터에서 서울에서 인천으로 가는 통근량은 정확히 몇 명이야?"
).text)
# 정답은 520 — 그런데 모델은 이 데이터를 알 리가 없다.

In [ ]:
# 한계 ② — 멈춰 있는 지식: 우리가 방금 만든 데이터를 모델이 알 수 없다.
print(llm.generate(
    "lab0 의 cities.json 에 들어 있는 18개 도시를 한글 이름으로 모두 나열해줘."
).text)
# 모델은 우리 파일을 본 적이 없다 — 비슷한 도시 이름을 추측해서 답할 것이다.

In [ ]:
# 한계 ③ — 행동 불가: 모델은 우리 파일을 열어볼 수 없다.
print(llm.generate(
    "labs/lab0_vibe_coding/data/flows.json 파일을 열어서 "
    "통근량이 가장 많은 구간을 알려줘."
).text)
# 정답은 서울→성남 540 — 모델은 파일을 *열어 본 척* 할 수도 있다 (그게 환각).

**관찰 포인트**

- 한계 ① 의 숫자가 우연히 맞을 수는 있지만, 모델은 **이유 없이** 답합니다 → 신뢰 못 함.
- 한계 ② 는 "모른다" 보다는 **그럴듯한 추측** 으로 나옵니다.
- 한계 ③ 은 파일을 본 듯 *연기* 할 수도 있습니다.

> **lab0 의 챗봇은 이 한계들을 어떻게 메웠을까?**
> 두 단계로 메웠습니다 — 둘 다 이번/다음 lab 의 주제입니다.
>
> 1. **시스템 프롬프트** 로 *역할과 원칙* 을 못 박는다 — "모르면 모른다고 해" (이번 lab)
> 2. **도구** 를 줘서 *데이터를 직접 보게* 한다 — `find_city`, `get_flow`, … (lab 2)
>
> 두 가지가 같이 작동해야 lab0 챗봇이 잘 답합니다. 1번만 있고 2번이 없으면
> 모델은 "모릅니다" 만 반복합니다. 2번만 있고 1번이 없으면 모델이 도구를
> 안 부르고 추측해 버립니다.


## 3. lab0 의 시스템 프롬프트 — 역할·원칙만

lab0 의 `src/lib/chat.js` 를 열어 보면 `SYSTEM_PROMPT` 는 *놀랍도록 짧습니다* —
데이터는 한 줄도 없습니다. 역할과 원칙만 있어요.

아래 `SYSTEM_PROMPT` 는 lab0 의 그 변수를 **그대로 옮긴 것** 입니다.


In [ ]:
# lab0/example_solution/src/lib/chat.js 의 SYSTEM_PROMPT 와 동일.
SYSTEM_PROMPT = """당신은 "수도권 통근 데이터 안내 도우미"입니다.
cities/flows 두 데이터에 대한 질문에 답하는 것이 당신의 유일한 임무입니다.

[원칙]
- 데이터에 없는 건 지어내지 않습니다. 모르는 건 모른다고 합니다.
- 데이터 조회·합계·최댓값은 반드시 제공된 도구를 호출해서 얻은 값을 인용합니다.
  머릿속으로 계산하거나 추측하지 않습니다.
- flows 는 방향이 있습니다. SEO→INC 와 INC→SEO 는 서로 다른 항목입니다.

[응답]
- 한국어, 2~4문장, 친근한 말투.
- 숫자를 답할 때는 그 값을 어떻게 얻었는지 한 줄로 함께 밝힙니다.
  예: "통근량이 가장 많은 구간은 서울→성남(540)입니다 — get_top_flows(1) 결과입니다."
- 사용자가 멀티스텝 질문(여러 도구가 필요한 질문)을 하면 도구를 순서대로 호출한 뒤
  마지막에 한 번에 정리해서 답합니다."""

print(f"시스템 프롬프트 길이: {len(SYSTEM_PROMPT)}자")
print("-- 데이터를 박지 않았다는 점에 주목 — cities/flows 단어는 있지만 그 *값* 은 없다 --")

### 3-1. 시스템 프롬프트가 *하는* 일

같은 질문을 — *없이* / *있이* — 던져 봅니다. **답하는 방식** 이 어떻게 달라지는지 보세요.


In [ ]:
q = "서울에서 인천으로 가는 통근량은?"

print("--- 시스템 프롬프트 없음 ---")
print(llm.generate(q).text)
print()
print("--- 시스템 프롬프트 있음 (lab0 와 동일) ---")
print(llm.generate(q, system=SYSTEM_PROMPT).text)

### 3-2. 시스템 프롬프트가 *못 하는* 일

시스템 프롬프트는 **태도** 를 바꿉니다 — "모르면 모른다고 해". 그러나 **데이터** 를 주지는 못합니다.

잘 작동한 케이스에서는 모델이 "도구를 호출해야 한다" 같은 답을 합니다. 또는 "모릅니다" 라고 합니다.
둘 다 lab0 처럼 정확한 **520** 을 알지는 못합니다 — *시스템 프롬프트만으로는*.

👉 그래서 lab0 는 **도구** 도 함께 줍니다. 그게 **lab 2** 입니다.


## 4. 도구 없이 "추론 → 행동" 끌어내기 (ReAct 맛보기)

진짜 도구 호출은 lab2 입니다. 지금은 시스템 프롬프트만으로 모델이 **생각(Thought)** 과
**행동 제안(Action)** 을 *글로* 쓰게 만들어 봅니다. 이것이 lab3 의 추론→행동→관찰 루프의 씨앗입니다.


In [ ]:
react_system = """You are a commute-data analysis agent for 18 cities in 수도권.
You cannot run tools yet, so instead THINK step by step and PROPOSE an action.
Reply in exactly this format, in Korean:

Thought: <무엇을 알아내야 하는지 한 줄>
Action: <호출하면 좋을 가상의 도구와 입력. 예: get_flow(origin="서울", dest="인천")>
"""

questions = [
    "서울에서 인천으로 가는 통근량은?",                # → get_flow
    "수원과 용인 사이 왕복 통근량은?",                 # → get_round_trip
    "통근량이 가장 많은 구간을 알고 그 두 도시 왕복도 알고 싶어.",  # → 멀티스텝!
]
for q in questions:
    print("Q:", q)
    print(llm.generate(q, system=react_system).text)
    print()

모델이 "어떤 도구를, 어떤 입력으로 부르면 좋을지" 를 *글로* 말합니다. 마지막 질문에서는
*도구를 두 번* 부르겠다는 계획을 세우기도 합니다.

- **lab 2** 에서는 이 *제안* 을 **진짜 함수 호출** 로 바꿉니다 (Gemini function calling).
- **lab 3** 에서는 그 호출을 *여러 번 반복* 하는 루프 (`runAgent`) 로 만듭니다.

둘이 합쳐지면 — 그게 바로 lab0 에서 실행되던 그 챗봇입니다.


## 🔧 TODO — 나만의 시스템 프롬프트 작성

위 `SYSTEM_PROMPT` 를 본보기로, **분석가 페르소나** 의 시스템 프롬프트를 직접 써 보세요.
*역할/말투/형식* 만 지정하고, *데이터는 박지 않습니다*.

요구사항:
- 한국어로 답한다
- 답은 **3문장 이내**
- 항상 끝에 "추가로 확인하면 좋을 분석" 한 가지를 제안한다


In [ ]:
# 🔧 TODO: my_system 문자열을 채우세요.
my_system = """"""   # <-- 여기에 작성

# --- 테스트 ---
test_q = "서울과 인천 사이 통근량 차이(서울→인천 - 인천→서울)는 왜 생길까?"
result = llm.generate(test_q, system=my_system or "You are a helpful assistant.")
print(result.text)

## 정리

- LLM 은 텍스트 생성기다 — 한계 ①②③ 을 lab0 데이터로 직접 확인했다.
- 시스템 프롬프트(`Instructions`)는 **태도** 를 바꾼다: "모르면 모른다고 해".
- 시스템 프롬프트만으로는 **데이터** 를 주지 못한다 → lab0 가 *도구* 를 같이 주는 이유.
- 프롬프트만으로도 "추론→행동" 형식을 끌어낼 수 있다 → lab2/lab3 의 씨앗.

**lab0 의 양파 다시 보기**

| 안쪽 → 바깥쪽 | lab0 파일 | 이번 lab | 다음 |
|---|---|---|---|
| LLM 호출 + 시스템 프롬프트 | `chat.js` | ✅ lab1 | |
| 도구 (function calling) | `tools.js` | | **lab2** |
| 루프 (멀티스텝) | `agent.js` | | lab3 |

**다음 — Session 2 (도구 사용)**
> 시스템 프롬프트가 *못 했던* 일을 도구가 한다 — 모델에게 "필요할 때 부를 함수 목록" 을 준다.
> 그러면 한계 ③(행동 불가)이 사라지고, 한계 ①(환각)도 크게 줄어든다.
